# Self-Healing Machine Learning System — Trial 1

## Objective
This experiment evaluates whether a machine-learning system can detect
performance degradation caused by artificially injected missing values,
apply an approved correction, retrain a candidate model, and automatically
decide whether to accept or reject the corrected model.

### Trial 1 Configuration
- Dataset: UCI Default of Credit Card Clients
- Problem: Binary Classification
- Baseline Model: Logistic Regression
- Comparison Model: TensorFlow/Keras Neural Network
- Failure Type: Artificially Injected Missing Values
- Failure Levels: 5%, 15%, 30%
- Correction: Median Imputation
- Primary Metric: F1 Score
- Additional Metrics: Recall, ROC-AUC, Accuracy

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 


In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [9]:
from pathlib import Path

data_dir = Path("data/raw")
candidates = sorted(data_dir.glob("*credit*card*clients*.xls*"))
if not candidates:
    raise FileNotFoundError(f"No dataset file found under {data_dir}")

DATA_PATH = str(candidates[0])
df = pd.read_excel(DATA_PATH, header=1)
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [10]:
print(df.columns.to_list())

['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_A

## 2. Feature and Target Definition

The `ID` column is excluded because it is a record identifier rather than
a meaningful predictive feature.

The target variable is `default payment next month`:

- 0 = No default
- 1 = Default

The remaining 23 variables are used as predictors.

In [12]:
# Define X and y
TARGET = "default payment next month"
X = df.drop(columns = ["ID", TARGET])
y = df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (30000, 23)
y shape: (30000,)


In [13]:
class_counts = y.value_counts().sort_index()
class_percentages = y.value_counts(normalize = True).sort_index()*100
class_summary = pd.DataFrame({
    "Count": class_counts,
    "Percentage": class_percentages
})
class_summary

,Count,Percentage
default payment next month,,
0,23364,77.88
1,6636,22.12


In [14]:
total_missing = X.isnull().sum().sum()
missing_percentage = (total_missing/X.size) * 100
print("Total missing values:", total_missing)
print(f"Overall missing percentage: {missing_percentage:.2f}%")

Total missing values: 0
Overall missing percentage: 0.00%


In [17]:
print(y.value_counts())

default payment next month
0    23364
1     6636
Name: count, dtype: int64


In [22]:
target = y
class_distribution = target.value_counts().sort_index()
class_percentage = target.value_counts(normalize=True).sort_index() * 100
class_summary = pd.DataFrame({
    "Count": class_distribution,
    "Percentage": class_percentage
})

class_summary

,Count,Percentage
default payment next month,,
0,23364,77.88
1,6636,22.12


In [23]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 23 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   LIMIT_BAL  30000 non-null  int64
 1   SEX        30000 non-null  int64
 2   EDUCATION  30000 non-null  int64
 3   MARRIAGE   30000 non-null  int64
 4   AGE        30000 non-null  int64
 5   PAY_0      30000 non-null  int64
 6   PAY_2      30000 non-null  int64
 7   PAY_3      30000 non-null  int64
 8   PAY_4      30000 non-null  int64
 9   PAY_5      30000 non-null  int64
 10  PAY_6      30000 non-null  int64
 11  BILL_AMT1  30000 non-null  int64
 12  BILL_AMT2  30000 non-null  int64
 13  BILL_AMT3  30000 non-null  int64
 14  BILL_AMT4  30000 non-null  int64
 15  BILL_AMT5  30000 non-null  int64
 16  BILL_AMT6  30000 non-null  int64
 17  PAY_AMT1   30000 non-null  int64
 18  PAY_AMT2   30000 non-null  int64
 19  PAY_AMT3   30000 non-null  int64
 20  PAY_AMT4   30000 non-null  int64
 21  PAY_AMT5   3

In [15]:
from sklearn.model_selection import train_test_split

In [ ]:
# first split: 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)

In [24]:
# Second split: divide remaining 30% equally
# 15% validation + 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

Training set: (21000, 23)
Validation set: (4500, 23)
Test set: (4500, 23)


In [25]:
# verification 
def show_class_distribution(name, y_data):
    counts = y_data.value_counts().sort_index()
    percentages = y_data.value_counts(normalize = True).sort_index() * 100
    print(f"\n{name}")
    print("-" * 30)

    for cls in counts.index:
        print(
            f"Class {cls}: "
            f"{counts[cls]} samples "
            f"({percentages[cls]:.2f}%)"
        )

show_class_distribution("Training Set", y_train)
show_class_distribution("Validation Set", y_val)
show_class_distribution("Test Set", y_test)


Training Set
------------------------------
Class 0: 16355 samples (77.88%)
Class 1: 4645 samples (22.12%)

Validation Set
------------------------------
Class 0: 3505 samples (77.89%)
Class 1: 995 samples (22.11%)

Test Set
------------------------------
Class 0: 3504 samples (77.87%)
Class 1: 996 samples (22.13%)


## 3. Healthy Baseline Model — Logistic Regression

Logistic Regression is used as the baseline model for Trial 1 because it
provides a simple, interpretable, and computationally efficient binary
classification baseline.

The model is trained on clean training data and evaluated on clean
validation data. These validation metrics represent the healthy baseline
against which future performance degradation and recovery will be measured.

Primary metric: F1 score

Additional metrics:
- Recall
- ROC-AUC
- Accuracy

In [26]:
from sklearn.preprocessing import StandardScaler

In [27]:
scaler = StandardScaler()

In [28]:
X_trained_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [29]:
scaler.fit_transform

<bound method TransformerMixin.fit_transform of StandardScaler()>

In [30]:
from sklearn.linear_model import LogisticRegression

In [ ]:
#train logistic regression
baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)
baseline_model.fit(X_trained_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [37]:
# validation predictions
# Keep class predictions and class probabilities separate for metric calculations.
y_val_pred = baseline_model.predict(X_val_scaled)
y_val_prob = baseline_model.predict_proba(X_val_scaled)[:, 1]


In [39]:
# probabilities are stored in y_val_prob; class labels remain in y_val_pred
print("Validation prediction shape:", y_val_pred.shape)
print("Validation probability shape:", y_val_prob.shape)


Validation prediction shape: (4500,)
Validation probability shape: (4500,)


In [35]:
#calculate healthy baseline metrics
from sklearn.metrics import (
    f1_score,
    recall_score,
    roc_auc_score,
    accuracy_score
)

In [40]:
baseline_f1 = f1_score(y_val, y_val_pred)
baseline_recall = recall_score(y_val, y_val_pred)
baseline_roc_auc = roc_auc_score(y_val, y_val_prob)
baseline_accuracy = accuracy_score(y_val, y_val_pred)

print("HEALTHY BASELINE — LOGISTIC REGRESSION")
print("-" * 45)
print(f"F1 Score : {baseline_f1:.4f}")
print(f"Recall   : {baseline_recall:.4f}")
print(f"ROC-AUC  : {baseline_roc_auc:.4f}")
print(f"Accuracy : {baseline_accuracy:.4f}")

HEALTHY BASELINE — LOGISTIC REGRESSION
---------------------------------------------
F1 Score : 0.3570
Recall   : 0.2402
ROC-AUC  : 0.7130
Accuracy : 0.8087


### Healthy Baseline Results

The Logistic Regression model established the following healthy validation
performance:

- F1 Score: 0.3570
- Recall: 0.2402
- ROC-AUC: 0.7130
- Accuracy: 0.8087

Although accuracy is relatively high, recall is substantially lower,
indicating that the model fails to identify many clients who actually
default. Therefore, F1 is retained as the primary metric because it provides
a more informative balance between precision and recall for the positive
(default) class.

These metrics represent the healthy reference state against which
performance degradation and subsequent recovery will be evaluated.

In [41]:
healthy_baseline = {
    "f1": baseline_f1,
    "recall": baseline_recall,
    "roc_auc": baseline_roc_auc,
    "accuracy": baseline_accuracy
}

healthy_baseline

{'f1': 0.3569828230022405,
 'recall': 0.24020100502512562,
 'roc_auc': 0.7130253263464255,
 'accuracy': 0.8086666666666666}

## 4. Controlled Failure Injection

Trial 1 evaluates the SHML system under artificially introduced missing-data
failures.

Three corrupted validation batches are created:

- Batch A: 5% missing values
- Batch B: 15% missing values
- Batch C: 30% missing values

Missing values are introduced randomly across predictor features while the
target variable remains unchanged.

The original validation dataset is preserved so that all corrupted batches
can be compared against the same healthy reference data.

A fixed random seed is used to make the experiment reproducible.

In [44]:
def inject_missing_values(X, missing_rate, random_state=42):
    """
    Randomly inject missing values into a copy of a feature DataFrame.

    Parameters
    ----------
    X : pandas.DataFrame
        Clean feature dataset.

    missing_rate : float
        Fraction of feature cells to replace with NaN.
        Example: 0.05 = 5%.

    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    pandas.DataFrame
        Corrupted copy of X.
    """

    X_corrupted = X.copy()
    rng = np.random.default_rng(random_state)
    total_cells = X_corrupted.size
    n_missing = int(total_cells * missing_rate)
    missing_positions = rng.choice(total_cells, size=n_missing, replace=False)

    # Convert DataFrame temporarily to float so it can contain NaN
    corrupted_array = X_corrupted.to_numpy(dtype=float)

    # Insert NaN values
    corrupted_array.flat[missing_positions] = np.nan

    # Convert back to DataFrame
    X_corrupted = pd.DataFrame(
        corrupted_array,
        columns=X.columns,
        index=X.index
    )

    return X_corrupted


In [45]:
#failure batches 
X_val_5 = inject_missing_values(
    X_val,
    missing_rate=0.05,
    random_state=42
)

X_val_15 = inject_missing_values(
    X_val,
    missing_rate=0.15,
    random_state=42
)

X_val_30 = inject_missing_values(
    X_val,
    missing_rate=0.30,
    random_state=42
)

In [46]:
#missingness monitor
def calculate_missing_rate(X):
    """
    Calculate the overall percentage of missing feature values.
    """

    missing_cells = X.isna().sum().sum()
    total_cells = X.size

    missing_rate = missing_cells / total_cells

    return missing_rate

In [47]:
print(f"Clean validation: {calculate_missing_rate(X_val):.2%}")
print(f"Batch A:          {calculate_missing_rate(X_val_5):.2%}")
print(f"Batch B:          {calculate_missing_rate(X_val_15):.2%}")
print(f"Batch C:          {calculate_missing_rate(X_val_30):.2%}")

Clean validation: 0.00%
Batch A:          5.00%
Batch B:          15.00%
Batch C:          30.00%


In [48]:
print(
    "Original X_val missing values:",
    X_val.isna().sum().sum()
)

print(
    "Original X_val missing rate:",
    f"{calculate_missing_rate(X_val):.2%}"
)

Original X_val missing values: 0
Original X_val missing rate: 0.00%


In [49]:
MISSINGNESS_THRESHOLD = 0.05

In [50]:
def missingness_alert(X, threshold=0.05):

    missing_rate = calculate_missing_rate(X)

    alert = missing_rate > threshold

    return {
        "missing_rate": missing_rate,
        "threshold": threshold,
        "alert": alert
    }

In [51]:
for name, dataset in [
    ("Clean", X_val),
    ("Batch A - 5%", X_val_5),
    ("Batch B - 15%", X_val_15),
    ("Batch C - 30%", X_val_30)
]:
    result = missingness_alert(dataset)

    print(
        f"{name:15} | "
        f"Missing: {result['missing_rate']:.2%} | "
        f"Alert: {result['alert']}"
    )

Clean           | Missing: 0.00% | Alert: False
Batch A - 5%    | Missing: 5.00% | Alert: False
Batch B - 15%   | Missing: 15.00% | Alert: True
Batch C - 30%   | Missing: 30.00% | Alert: True


## 5. Failure Detection

The SHML system evaluates incoming batches for data-quality failures before
model inference.

For Trial 1, missingness is the monitored failure condition. Logistic
Regression cannot directly process NaN values, so a sufficiently corrupted
batch may also cause an inference failure.

The failure detector therefore records:

1. Missing-value rate
2. Missingness threshold status
3. Whether model inference succeeds

A detected failure can trigger the approved healing strategy.

In [55]:
def check_model_inference(model, scaler, X_batch):
    try:
        X_scaled = scaler.transform(X_batch)
        prediction = model.predict(X_scaled)

        return {
            "inference_success": True,
            "error": None,
            "prediction_shape": prediction.shape
        }
    except ValueError as error:
        return {
            "inference_success": False,
            "error": str(error),
            "prediction_shape": None
        }


In [56]:
# testing all four datasets
batches = {
    "Clean": X_val,
    "5% Missing": X_val_5,
    "15% Missing": X_val_15,
    "30% Missing": X_val_30
}

for name, batch in batches.items():
    result = check_model_inference(
        baseline_model,
        scaler,
        batch
    )
    
    print(
            f"{name:12} | "
            f"Inference success: {result['inference_success']}"
        )

Clean        | Inference success: True
5% Missing   | Inference success: False
15% Missing  | Inference success: False
30% Missing  | Inference success: False


In [57]:
#healing the pipeline 
X_val_15.median()

LIMIT_BAL    140000.0
SEX               2.0
EDUCATION         2.0
MARRIAGE          2.0
AGE              34.0
PAY_0             0.0
PAY_2             0.0
PAY_3             0.0
PAY_4             0.0
PAY_5             0.0
PAY_6             0.0
BILL_AMT1     22701.0
BILL_AMT2     21790.0
BILL_AMT3     20183.0
BILL_AMT4     18988.0
BILL_AMT5     18087.0
BILL_AMT6     17002.0
PAY_AMT1       2077.0
PAY_AMT2       2000.0
PAY_AMT3       1686.5
PAY_AMT4       1420.0
PAY_AMT5       1500.0
PAY_AMT6       1495.5
dtype: float64

In [58]:
from sklearn.impute import SimpleImputer

In [59]:
median_imputer = SimpleImputer(strategy = "median")

In [60]:
median_imputer.fit(X_train)

,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


In [61]:
#now healing the 3 corrupted validation batches 
X_val_5_healed = median_imputer.transform(X_val_5)
X_val_15_healed = median_imputer.transform(X_val_15)
X_val_30_healed = median_imputer.transform(X_val_30)

In [62]:
#verify
print("Remaining NaNs after healing:")

print(
    "5% batch:",
    np.isnan(X_val_5_healed).sum()
)

print(
    "15% batch:",
    np.isnan(X_val_15_healed).sum()
)

print(
    "30% batch:",
    np.isnan(X_val_30_healed).sum()
)

Remaining NaNs after healing:
5% batch: 0
15% batch: 0
30% batch: 0


In [63]:
# evaluate healed batches 
from sklearn.metrics import(f1_score, recall_score, roc_auc_score, accuracy_score)

In [66]:
def evaluate_healed_batch(model, scaler, X_healed, y_true):
    """
    Evaluate an imputed batch using the approved baseline model.
    """
    
    #Apply the scaler learned from cleaned training data
    X_scaled = scaler.transform(X_healed)
    # class predictions
    y_pred = model.predict(X_scaled)
    #probability predictions for ROC-AUC
    y_prob = model.predict_proba(X_scaled)[:, 1]
    
    return{
        "f1": f1_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred)
    }

In [67]:
healed_5_metrics = evaluate_healed_batch(
    baseline_model,
    scaler,
    X_val_5_healed,
    y_val
)

healed_15_metrics = evaluate_healed_batch(
    baseline_model,
    scaler,
    X_val_15_healed,
    y_val
)

healed_30_metrics = evaluate_healed_batch(
    baseline_model,
    scaler,
    X_val_30_healed,
    y_val
)

/opt/anaconda3/envs/shml/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/shml/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/shml/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [ ]:
results = pd.DataFrame({
    "Healthy": healthy_baseline,
    "5% Missing + Healing": healed_5_metrics,
    "15% Missing + Healing": healed_15_metrics,
    "30% Missing + Healing": healed_30_metrics
}).T
results


,f1,recall,roc_auc,accuracy
Healthy,0.356983,0.240201,0.713025,0.808667
5% Missing + Healing,0.336634,0.222111,0.708736,0.806444
15% Missing + Healing,0.300787,0.191960,0.701598,0.802667
30% Missing + Healing,0.241004,0.144724,0.689380,0.798444


In [72]:
def compare_to_healthy(healthy_f1, healed_f1):
    absolute_change = healed_f1 - healthy_f1
    relative_change = (
        (healed_f1 - healthy_f1)/healthy_f1
    ) * 100
    return {
        "absolute_f1_change": absolute_change,
        "relative_f1_change_pct": relative_change
    }

In [73]:
for name, metrics in [
    ("5%", healed_5_metrics),
    ("15%", healed_15_metrics),
    ("30%", healed_30_metrics)
]:
    comparison = compare_to_healthy(
        healthy_baseline["f1"],
        metrics["f1"]
    )

    print(f"\n{name} Missingness")
    print(
        f"Healed F1: "
        f"{metrics['f1']:.4f}"
    )
    print(
        f"Absolute change from healthy: "
        f"{comparison['absolute_f1_change']:.4f}"
    )
    print(
        f"Relative change from healthy: "
        f"{comparison['relative_f1_change_pct']:.2f}%"
    )


5% Missingness
Healed F1: 0.3366
Absolute change from healthy: -0.0203
Relative change from healthy: -5.70%

15% Missingness
Healed F1: 0.3008
Absolute change from healthy: -0.0562
Relative change from healthy: -15.74%

30% Missingness
Healed F1: 0.2410
Absolute change from healthy: -0.1160
Relative change from healthy: -32.49%


In [74]:
SAFETY_TOLERANCE = 0.10

minimum_acceptable_f1 = (
    healthy_baseline["f1"] * (1 - SAFETY_TOLERANCE)
)

print(f"Healthy F1: {healthy_baseline['f1']:.4f}")
print(f"Minimum acceptable F1: {minimum_acceptable_f1:.4f}")

Healthy F1: 0.3570
Minimum acceptable F1: 0.3213


After correction, accept the repaired batch/model state only if inference succeeds and F1 remains within 10% of the healthy baseline. Otherwise, reject the recovery as insufficient and retain the current approved model/state.

In [81]:
# Building Validator
def validate_recovery(
    healthy_f1,
    healed_metrics,
    safety_tolerance=0.10
):
    """
    Validate whether the healed batch remains within
    the allowed performance tolerance of the healthy baseline.
    """

    healed_f1 = healed_metrics["f1"]

    # Minimum F1 allowed after healing
    minimum_acceptable_f1 = healthy_f1 * (1 - safety_tolerance)

    # Relative performance loss
    relative_drop = (healthy_f1 - healed_f1) / healthy_f1

    # Accept only if healed F1 remains within safety tolerance
    accepted = healed_f1 >= minimum_acceptable_f1

    return {
        "healthy_f1": healthy_f1,
        "healed_f1": healed_f1,
        "safety_tolerance": safety_tolerance,
        "minimum_acceptable_f1": minimum_acceptable_f1,
        "relative_drop": relative_drop,
        "accepted": accepted
    }


In [82]:
validation_results = {}

for name, metrics in [
    ("5%", healed_5_metrics),
    ("15%", healed_15_metrics),
    ("30%", healed_30_metrics)
]:

    result = validate_recovery(
        healthy_baseline["f1"],
        metrics
    )

    validation_results[name] = result

    print(f"\n{name} Missingness")
    print("-" * 30)
    print(f"Healthy F1:     {result['healthy_f1']:.4f}")
    print(f"Healed F1:      {result['healed_f1']:.4f}")
    print(
        f"Minimum F1:     "
        f"{result['minimum_acceptable_f1']:.4f}"
    )
    print(
        f"Relative drop:  "
        f"{result['relative_drop']:.2%}"
    )
    print(
        "Decision:       "
        + ("ACCEPT" if result["accepted"] else "REJECT")
    )


5% Missingness
------------------------------
Healthy F1:     0.3570
Healed F1:      0.3366
Minimum F1:     0.3213
Relative drop:  5.70%
Decision:       ACCEPT

15% Missingness
------------------------------
Healthy F1:     0.3570
Healed F1:      0.3008
Minimum F1:     0.3213
Relative drop:  15.74%
Decision:       REJECT

30% Missingness
------------------------------
Healthy F1:     0.3570
Healed F1:      0.2410
Minimum F1:     0.3213
Relative drop:  32.49%
Decision:       REJECT


In [83]:
print(
    healthy_baseline["f1"] * (1 - SAFETY_TOLERANCE)
)

0.32128454070201645


In [84]:
test_result = validate_recovery(
    healthy_f1=healthy_baseline["f1"],
    healed_metrics=healed_15_metrics,
    safety_tolerance=SAFETY_TOLERANCE
)

test_result

{'healthy_f1': 0.3569828230022405,
 'healed_f1': 0.30078740157480316,
 'safety_tolerance': 0.1,
 'minimum_acceptable_f1': 0.32128454070201645,
 'relative_drop': 0.15741771818271671,
 'accepted': False}

### Recovery Validation

Median imputation successfully removed all injected missing values and
restored the ability of the Logistic Regression pipeline to perform
inference. However, successful data repair did not guarantee complete
predictive recovery.

The 5% missingness condition produced a healed F1 of 0.3366, representing
a 5.70% decline relative to the healthy baseline F1 of 0.3570. This remained
within the preliminary 10% safety tolerance and was therefore accepted.

At 15% missingness, healed F1 decreased to 0.3008, a 15.74% relative decline.
At 30% missingness, healed F1 decreased to 0.2410, a 32.49% relative decline.
Both exceeded the allowed performance degradation and were therefore
rejected.

These results demonstrate that removing the detected data-quality failure
is not sufficient by itself. The SHML system must validate predictive
performance after correction before accepting a recovered state.

In [85]:
validation_results = {}

for name, metrics in [
    ("5%", healed_5_metrics),
    ("15%", healed_15_metrics),
    ("30%", healed_30_metrics)
]:

    result = validate_recovery(
        healthy_f1=healthy_baseline["f1"],
        healed_metrics=metrics,
        safety_tolerance=SAFETY_TOLERANCE
    )

    validation_results[name] = result

    print(f"\n{name} Missingness")
    print("-" * 30)
    print(f"Healthy F1:     {result['healthy_f1']:.4f}")
    print(f"Healed F1:      {result['healed_f1']:.4f}")
    print(f"Minimum F1:     {result['minimum_acceptable_f1']:.4f}")
    print(f"Relative drop:  {result['relative_drop']:.2%}")
    print(
        "Decision:       "
        + ("ACCEPT" if result["accepted"] else "REJECT")
    )


5% Missingness
------------------------------
Healthy F1:     0.3570
Healed F1:      0.3366
Minimum F1:     0.3213
Relative drop:  5.70%
Decision:       ACCEPT

15% Missingness
------------------------------
Healthy F1:     0.3570
Healed F1:      0.3008
Minimum F1:     0.3213
Relative drop:  15.74%
Decision:       REJECT

30% Missingness
------------------------------
Healthy F1:     0.3570
Healed F1:      0.2410
Minimum F1:     0.3213
Relative drop:  32.49%
Decision:       REJECT


## 6. Automated Self-Healing Decision Loop

The individual failure detection, correction, and validation components are
combined into a single automated decision process.

For each incoming batch, the SHML system:

1. Measures the missing-value rate.
2. Determines whether the current model can perform inference.
3. Identifies a missing-data or hard-inference failure.
4. Applies median imputation when healing is required.
5. Evaluates predictive performance after correction.
6. Compares healed F1 against the healthy baseline.
7. Accepts the recovered state only when it satisfies the safety requirement.
8. Rejects insufficient recovery and retains the approved model.

In [ ]:
#Building automatics SHML healing loop

In [86]:
def run_shml_trial(
    X_batch,
    y_batch,
    model,
    scaler,
    imputer,
    healthy_baseline,
    missingness_threshold=0.05,
    safety_tolerance=0.10
):
    """
    Run the Trial 1 SHML detection → correction → validation loop.
    """

    # ----------------------------------
    # 1. Monitor incoming data
    # ----------------------------------

    missing_rate = calculate_missing_rate(X_batch)

    missing_alert = missing_rate > missingness_threshold

    # ----------------------------------
    # 2. Check whether current model
    #    can process the batch
    # ----------------------------------

    inference_check = check_model_inference(
        model,
        scaler,
        X_batch
    )

    inference_success = inference_check["inference_success"]

    # ----------------------------------
    # 3. Determine whether healing
    #    is required
    # ----------------------------------

    healing_required = (
        missing_alert or
        not inference_success
    )

    # Healthy / usable batch
    if not healing_required:

        metrics = evaluate_healed_batch(
            model,
            scaler,
            X_batch,
            y_batch
        )

        return {
            "missing_rate": missing_rate,
            "missing_alert": missing_alert,
            "inference_success_before_healing": inference_success,
            "healing_applied": False,
            "correction": None,
            "metrics": metrics,
            "decision": "NO HEALING REQUIRED"
        }

    # ----------------------------------
    # 4. Apply approved correction
    # ----------------------------------

    X_healed = imputer.transform(X_batch)

    # ----------------------------------
    # 5. Evaluate repaired batch
    # ----------------------------------

    healed_metrics = evaluate_healed_batch(
        model,
        scaler,
        X_healed,
        y_batch
    )

    # ----------------------------------
    # 6. Validate recovery
    # ----------------------------------

    validation = validate_recovery(
        healthy_f1=healthy_baseline["f1"],
        healed_metrics=healed_metrics,
        safety_tolerance=safety_tolerance
    )

    # ----------------------------------
    # 7. Accept / Reject
    # ----------------------------------

    if validation["accepted"]:
        decision = "ACCEPT RECOVERY"
    else:
        decision = "REJECT RECOVERY — KEEP APPROVED MODEL / ESCALATE"

    return {
        "missing_rate": missing_rate,
        "missing_alert": missing_alert,
        "inference_success_before_healing": inference_success,
        "healing_applied": True,
        "correction": "Median Imputation",
        "healed_metrics": healed_metrics,
        "validation": validation,
        "decision": decision
    }

In [87]:
trial_results = {}

for name, batch in [
    ("5%", X_val_5),
    ("15%", X_val_15),
    ("30%", X_val_30)
]:

    result = run_shml_trial(
        X_batch=batch,
        y_batch=y_val,
        model=baseline_model,
        scaler=scaler,
        imputer=median_imputer,
        healthy_baseline=healthy_baseline,
        missingness_threshold=0.05,
        safety_tolerance=0.10
    )

    trial_results[name] = result

/opt/anaconda3/envs/shml/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/shml/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/shml/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [88]:
for name, result in trial_results.items():

    print(f"\n{name} MISSINGNESS")
    print("=" * 50)

    print(
        f"Missing rate:       "
        f"{result['missing_rate']:.2%}"
    )

    print(
        f"Missing alert:      "
        f"{result['missing_alert']}"
    )

    print(
        f"Inference before:   "
        f"{result['inference_success_before_healing']}"
    )

    print(
        f"Healing applied:    "
        f"{result['healing_applied']}"
    )

    print(
        f"Correction:         "
        f"{result['correction']}"
    )

    if result["healing_applied"]:

        print(
            f"Healed F1:          "
            f"{result['healed_metrics']['f1']:.4f}"
        )

        print(
            f"Relative F1 drop:   "
            f"{result['validation']['relative_drop']:.2%}"
        )

    print(
        f"Decision:           "
        f"{result['decision']}"
    )


5% MISSINGNESS
Missing rate:       5.00%
Missing alert:      False
Inference before:   False
Healing applied:    True
Correction:         Median Imputation
Healed F1:          0.3366
Relative F1 drop:   5.70%
Decision:           ACCEPT RECOVERY

15% MISSINGNESS
Missing rate:       15.00%
Missing alert:      True
Inference before:   False
Healing applied:    True
Correction:         Median Imputation
Healed F1:          0.3008
Relative F1 drop:   15.74%
Decision:           REJECT RECOVERY — KEEP APPROVED MODEL / ESCALATE

30% MISSINGNESS
Missing rate:       30.00%
Missing alert:      True
Inference before:   False
Healing applied:    True
Correction:         Median Imputation
Healed F1:          0.2410
Relative F1 drop:   32.49%
Decision:           REJECT RECOVERY — KEEP APPROVED MODEL / ESCALATE


## Trial 1 Preliminary Findings

The healthy Logistic Regression model achieved an F1 score of 0.3570.

Controlled missing-value failures were introduced at 5%, 15%, and 30%
levels. Because the baseline Logistic Regression pipeline could not process
raw NaN values, missingness resulted in an inference failure and triggered
the healing mechanism.

Median imputation successfully removed all missing values and restored
technical inference capability at all three corruption levels.

However, predictive recovery decreased as missingness increased:

- 5% missingness: F1 = 0.3366, 5.70% below baseline — ACCEPTED
- 15% missingness: F1 = 0.3008, 15.74% below baseline — REJECTED
- 30% missingness: F1 = 0.2410, 32.49% below baseline — REJECTED

Using a preliminary safety tolerance of 10% relative F1 degradation, only
the 5% corruption condition achieved acceptable predictive recovery.

These results demonstrate that correcting a data-quality problem does not
necessarily restore model performance. Therefore, an SHML system requires
post-correction validation and a safety mechanism that rejects insufficient
recovery.

Rejected recovery states are not promoted. The approved model configuration
is retained and the failure is escalated for an alternative correction
strategy or further investigation.